# Austin Crime Analysis
## Exploratory Data Analysis

This notebook explores crime incident data from Austin, Texas.

Goals:
- Understand the structure of this dataset
- Clean key fields
- Create time-based features
- Identify early crime patterns

In [1]:
# verify notebook is running in the correct environment
import sys
print(sys.executable)

c:\Users\diana\anaconda3\envs\austin_crime_env\python.exe


## Import Libraries

In [2]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
import plotly.express as px

## Load datasets

In [3]:
# Load crime data
df = pd.read_csv('../data/Crime_Reports.csv')

# check
df.head()
#df.info()

,Incident Number,Highest Offense Description,Highest Offense Code,Family Violence,Occurred Date Time,Occurred Date,Occurred Time,Report Date Time,Report Date,Report Time,Location Type,Council District,APD Sector,APD District,Clearance Status,Clearance Date,UCR Category,Category Description,Census Block Group
0,20055061757,BURGLARY OF RESIDENCE,500,N,11/27/2005 14:00,11/27/2005,1400.0,11/29/2005 18:21,11/29/2005,1821.0,RESIDENCE / HOME,7.0,AD,3,N,12/14/2005,220,Burglary,4.530452e+09
1,20053261821,FAILURE TO IDENTIFY,2707,N,11/22/2005 20:20,11/22/2005,2020.0,11/22/2005 20:20,11/22/2005,2020.0,HWY / ROAD / ALLEY/ STREET/ SIDEWALK,3.0,HE,2,C,12/03/2005,NaN,NaN,4.530023e+09
2,20055061758,CRIMINAL MISCHIEF,1400,N,11/29/2005 15:00,11/29/2005,1500.0,11/29/2005 18:22,11/29/2005,1822.0,RESIDENCE / HOME,7.0,ID,7,NaN,12/16/2005,NaN,NaN,4.530015e+09
3,20055061759,CRIMINAL MISCHIEF,1400,N,11/29/2005 14:00,11/29/2005,1400.0,11/29/2005 18:24,11/29/2005,1824.0,RESIDENCE / HOME,10.0,AD,7,NaN,NaN,NaN,NaN,4.530327e+09
4,20055061760,ASSAULT BY CONTACT,902,N,11/29/2005 11:00,11/29/2005,1100.0,11/29/2005 18:33,11/29/2005,1833.0,DRUG STORE / DOCTOR'S OFFICE / HOSPITAL,9.0,BA,5,NaN,12/27/2005,NaN,NaN,4.530002e+09


In [4]:
# Load APD district polygons
districts = pd.read_csv("../data/Austin_Police_Department_Districts_20260309.csv")

# check
districts.head()
districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   the_geom              84 non-null     object
 1   OBJECTID              84 non-null     int64 
 2   DISTRICT_NAME         84 non-null     object
 3   SORT_ORDER            84 non-null     int64 
 4   BATTALION_ID          84 non-null     int64 
 5   JURISDICTION_ID       84 non-null     int64 
 6   COLOR                 84 non-null     object
 7   CODE                  84 non-null     int64 
 8   EXTERNAL_KEY          84 non-null     int64 
 9   BATTALION_CODE        84 non-null     object
 10  SECTOR_NAME           84 non-null     object
 11  INPUT_DATE            84 non-null     object
 12  MODIFIED_DATE         84 non-null     object
 13  INPUT_BY              84 non-null     object
 14  MODIFIED_BY           84 non-null     object
 15  BUREAU_NAME           76 non-null     obje

## Clean up Crime_Reports.csv

In [5]:
# Standardize column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2625848 entries, 0 to 2625847
Data columns (total 19 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   incident_number              int64  
 1   highest_offense_description  object 
 2   highest_offense_code         int64  
 3   family_violence              object 
 4   occurred_date_time           object 
 5   occurred_date                object 
 6   occurred_time                float64
 7   report_date_time             object 
 8   report_date                  object 
 9   report_time                  float64
 10  location_type                object 
 11  council_district             float64
 12  apd_sector                   object 
 13  apd_district                 object 
 14  clearance_status             object 
 15  clearance_date               object 
 16  ucr_category                 object 
 17  category_description         object 
 18  census_block_group           float64
dtype

In [6]:
# convert to timestamp
df["occurred_date_time"] = pd.to_datetime(
    df["occurred_date_time"], 
    errors="coerce"
)

df["report_date_time"] = pd.to_datetime(
    df["report_date_time"], 
    errors="coerce"
)

# drop rows where the occurance time is missing
df = df.dropna(subset=["occurred_date_time"])

In [7]:
# create time features
df["year"] = df["occurred_date_time"].dt.year
df["month"] = df["occurred_date_time"].dt.month
df["hour"] = df["occurred_date_time"].dt.hour
df["weekday"] = df["occurred_date_time"].dt.day_name()

In [8]:
# calculate reporting delay (How long after a crime it was reported to the police?)
df["reporting_delay"] = (df["report_date_time"] - df["occurred_date_time"]).dt.total_seconds() / 3600

In [9]:
# Dataset goes back to 2006, so reduce size of data and filter to recent years for better visuals on dashboard -> 2022 to present day
df = df[df["year"] >= 2022]

# check
df.shape

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 363962 entries, 2250743 to 2625844
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   incident_number              363962 non-null  int64         
 1   highest_offense_description  363962 non-null  object        
 2   highest_offense_code         363962 non-null  int64         
 3   family_violence              363962 non-null  object        
 4   occurred_date_time           363962 non-null  datetime64[ns]
 5   occurred_date                363962 non-null  object        
 6   occurred_time                363962 non-null  float64       
 7   report_date_time             363962 non-null  datetime64[ns]
 8   report_date                  363962 non-null  object        
 9   report_time                  363962 non-null  float64       
 10  location_type                363921 non-null  object        
 11  council_district        

In [10]:
# remove missing districts
df = df.dropna(subset=["apd_district"])

In [11]:
# drop unnecessary columns to reduce memory usage & keep only what we might need for the dashboard
df = df[[
    "incident_number",
    "highest_offense_description",
    "family_violence",
    "occurred_date_time",
    "report_date_time",
    "year",
    "month",
    "hour",
    "weekday",
    "location_type",
    "council_district",
    "apd_sector",
    "apd_district",
    "reporting_delay",
    "ucr_category",
    "category_description" ]]

# check
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 363000 entries, 2250743 to 2625844
Data columns (total 16 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   incident_number              363000 non-null  int64         
 1   highest_offense_description  363000 non-null  object        
 2   family_violence              363000 non-null  object        
 3   occurred_date_time           363000 non-null  datetime64[ns]
 4   report_date_time             363000 non-null  datetime64[ns]
 5   year                         363000 non-null  int32         
 6   month                        363000 non-null  int32         
 7   hour                         363000 non-null  int32         
 8   weekday                      363000 non-null  object        
 9   location_type                362959 non-null  object        
 10  council_district             360738 non-null  float64       
 11  apd_sector              

In [12]:
# save the clean crime report data set
df.to_csv("../data/austin_crime_reports_clean.csv", index=False)

### Quick EDA

In [13]:
# crime by hour
fig = px.histogram(df, x="hour", nbins=24, title="Crime Occurrences by Hour")
fig.show()

In [14]:
# crime by district
district_counts = (
    df.groupby("apd_district").size().reset_index(name="crime_count")
)

fig = px.bar(district_counts, x="apd_district", y="crime_count", title="Crime Occurrences by APD District")
fig.show()

In [15]:
filtered_rows = df.loc[(df["apd_sector"] == "88")]
print(filtered_rows.head())


         incident_number       highest_offense_description family_violence  \
2251744      20225000387  ASSAULT BY THREAT-FAM/DATING VIO               Y   
2252092        202260816                FAMILY DISTURBANCE               N   
2252830      20225000801               DAMAGE CITY VEHICLE               N   
2253339      20225001046               BURGLARY OF VEHICLE               N   
2253839      20225001247         THEFT-NO SUSPECT/FOLLOWUP               N   

         occurred_date_time    report_date_time  year  month  hour    weekday  \
2251744 2022-01-05 11:46:00 2022-01-05 11:46:00  2022      1    11  Wednesday   
2252092 2022-01-06 15:31:00 2022-01-06 15:31:00  2022      1    15   Thursday   
2252830 2022-01-10 03:55:00 2022-01-10 07:57:00  2022      1     3     Monday   
2253339 2022-01-04 11:30:00 2022-01-11 17:21:00  2022      1    11    Tuesday   
2253839 2022-01-13 01:30:00 2022-01-13 10:00:00  2022      1     1   Thursday   

                                           l

## Clean the APD Districts.csv

In [16]:
# standarize column names
districts.columns = districts.columns.str.lower().str.replace(' ', '_')
districts.info()
districts.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   the_geom              84 non-null     object
 1   objectid              84 non-null     int64 
 2   district_name         84 non-null     object
 3   sort_order            84 non-null     int64 
 4   battalion_id          84 non-null     int64 
 5   jurisdiction_id       84 non-null     int64 
 6   color                 84 non-null     object
 7   code                  84 non-null     int64 
 8   external_key          84 non-null     int64 
 9   battalion_code        84 non-null     object
 10  sector_name           84 non-null     object
 11  input_date            84 non-null     object
 12  modified_date         84 non-null     object
 13  input_by              84 non-null     object
 14  modified_by           84 non-null     object
 15  bureau_name           76 non-null     obje

,the_geom,objectid,district_name,sort_order,battalion_id,jurisdiction_id,color,code,external_key,battalion_code,...,input_date,modified_date,input_by,modified_by,bureau_name,patrol_area,command_phone_number,primary_key,shape__area,shape__length
0,MULTIPOLYGON (((-97.647680568458 30.1864621984...,74,LND,0,518,145,"16,776,960",1333,1333,APT2,...,2014 Dec 22 11:31:16 AM,2016 May 27 01:55:16 PM,bmartinlimuel,ap7657,NaN,ABIA,NaN,79,"13,070,668.4453125","28,596.7618210815"
1,MULTIPOLYGON (((-97.703440677552 30.2249304298...,48,HENRY 4,1,205,59,"16,711,808",1300,1300,H1,...,2014 Dec 22 11:31:16 AM,2018 Oct 09 01:41:54 PM,bmartinlimuel,APD_ADMIN,SOUTH,SOUTHCENTRAL,512-974-8106,65,"45,233,992.484375","27,115.48762932911"
2,MULTIPOLYGON (((-97.783202958299 30.4512802395...,9,ADAM 5,1,21,6,"16,744,448",1272,1272,A1,...,2018 Oct 09 01:40:26 PM,2018 Oct 09 01:40:26 PM,APD_ADMIN,APD_ADMIN,NORTH,NORTHWEST,512-974-5500,6,"212,761,556.19726563","87,887.68396542221"
3,MULTIPOLYGON (((-97.753551595881 30.2387933972...,71,DAVID 2,1,164,12,"16,711,680",1284,1284,D1,...,2014 Dec 22 11:31:16 AM,2018 Sep 06 08:39:11 AM,bmartinlimuel,AP7657,SOUTH,SOUTHWEST,512-974-8100,35,"88,043,902.875","40,709.31711745582"
4,MULTIPOLYGON (((-97.611506162118 30.3978290786...,20,EDWARD 8,1,174,13,"16,711,680",1291,1291,E1,...,2018 Sep 20 10:49:31 AM,2018 Oct 09 01:44:33 PM,APD_ADMIN,APD_ADMIN,CENTRAL,NORTHEAST,512-974-5500,50,"30,886,850.02734375","36,534.23134356716"


In [17]:
# keep only useful columns for dashboard and mapping
districts = districts[[
"the_geom",
"district_name",
"sector_name",
"bureau_name",
"patrol_area"
]]

# check
districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   the_geom       84 non-null     object
 1   district_name  84 non-null     object
 2   sector_name    84 non-null     object
 3   bureau_name    76 non-null     object
 4   patrol_area    84 non-null     object
dtypes: object(5)
memory usage: 3.4+ KB


In [18]:
# convert geometry
districts["geometry"] = districts["the_geom"].apply(wkt.loads)

# convert to GeoDataFrame
districts_gdf = gpd.GeoDataFrame(
    districts, 
    geometry="geometry",
    crs="EPSG:4326" )


In [19]:
# clean district names
districts_gdf["district_name_clean"] = districts_gdf["district_name"].str.replace(r'\s+\d+$', '', regex=True).str.strip()

# clean sector names
districts_gdf["sector_name_clean"] = districts_gdf["sector_name"].str.strip().str.title()

# clean bureau names
districts_gdf["bureau_name_clean"] = districts_gdf["bureau_name"].str.strip().str.title()

In [20]:
# keep only necessary columns
final_cols = [
    "district_name_clean",
    "sector_name_clean",
    "bureau_name_clean",
    "patrol_area",
    "geometry"
]

districts_gdf_clean = districts_gdf[final_cols]

In [21]:
# check for duplicates or missing geometry
duplicates = districts_gdf_clean['district_name_clean'][districts_gdf_clean['district_name_clean'].duplicated()]
print("Duplicates:", duplicates)

missing_geom = districts_gdf_clean['geometry'].isnull().sum()
print("Missing geometry:", missing_geom)

Duplicates: 6       DAVID
7        ADAM
10       ADAM
11      DAVID
13      DAVID
15       ADAM
16       ADAM
17      DAVID
18      FRANK
19      FRANK
20        IDA
22      HENRY
25       ADAM
26      HENRY
27      BAKER
28      HENRY
29     GEORGE
30        IDA
33      BAKER
34       ADAM
35      FRANK
36        IDA
39      BAKER
40     EDWARD
42    CHARLIE
44       ADAM
45     EDWARD
46      FRANK
47    CHARLIE
48    CHARLIE
49        IDA
50        IDA
51      BAKER
52    CHARLIE
53    CHARLIE
54     GEORGE
55    CHARLIE
56      FRANK
57    CHARLIE
58      FRANK
59      DAVID
61      DAVID
62      HENRY
63        LND
64        IDA
65        IDA
68     EDWARD
69      HENRY
71      BAKER
72      DAVID
73      BAKER
75      HENRY
76      HENRY
77     EDWARD
78      FRANK
79     EDWARD
80     EDWARD
81      BAKER
83     EDWARD
Name: district_name_clean, dtype: object
Missing geometry: 0


##### I had some rows with the same district_name_clean, but no missing geometry. So to prevent overlapping or duplicated areas when mapping, I'll dissolve polygons by using cleaned district names - having one polygon per district 

In [22]:
# dissolve polygons by district name to ensure we have one polygon per district
districts_gdf_clean = districts_gdf_clean.dissolve(by="district_name_clean", as_index=False)

# check
print(districts_gdf_clean[['district_name_clean', "geometry"]])

   district_name_clean                                           geometry
0                A1 UT  POLYGON ((-97.72448 30.38274, -97.72483 30.381...
1                 ADAM  MULTIPOLYGON (((-97.8644 30.35886, -97.86428 3...
2                  AOA  POLYGON ((-97.67427 30.21814, -97.67424 30.218...
3              B1 UT-1  POLYGON ((-97.73355 30.28907, -97.73319 30.289...
4              B1 UT-2  POLYGON ((-97.73575 30.29195, -97.73531 30.291...
5                B2 UT  POLYGON ((-97.7833 30.29144, -97.78293 30.2912...
6                BAKER  POLYGON ((-97.72006 30.2942, -97.7202 30.29397...
7        BAKER PARK PD  POLYGON ((-97.83495 30.33043, -97.83532 30.329...
8              C8 UT-1  POLYGON ((-97.72216 30.28373, -97.72207 30.283...
9              C8 UT-2  POLYGON ((-97.72799 30.27865, -97.72808 30.278...
10             C8 UT-3  POLYGON ((-97.72284 30.28511, -97.72275 30.285...
11             CHARLIE  MULTIPOLYGON (((-97.63197 30.27139, -97.63198 ...
12               DAVID  MULTIPOLYGON (

In [23]:
# save districts dataset as a GEOJSON for mapping
districts_gdf_clean.to_file("../data/apd_districts_clean.geojson", driver="GeoJSON")